# TokenRouter / Kimi K3 — free-tier validation

Checks whether the "50 million free tokens for Kimi K3" offer is real and
usable for MUFASA.

**Why this matters.** Classifying 6,000 papers locally means fighting a
heavily quantised model on Kaggle. A frontier model over an API would be
faster and better — and it is allowed: only the *shipped* model must run
offline on llama.cpp. Corpus classification and Observation extraction are
build-plane work, where the architecture already assumes a frontier model.

**What is unverified.** Kimi K3 is real (Moonshot AI, 2.8T MoE, open weights
released 26 July 2026). TokenRouter itself does not appear in independent
search results, so nothing about the 50M figure can be confirmed from
outside. That is what the cells below are for.

Run top to bottom. Cell 4 is the one that decides it.

## 1. Install the SDK

In [1]:
# %pip install -q -U openai

## 2. API key

Entered at runtime with `getpass`, so it is never echoed and never saved into
this notebook. **Do not paste the key into a cell** — this file is committed.

In [ ]:
import os
from pathlib import Path

# .env lives next to the data files and is gitignored, so the key never
# reaches a commit. Add:  TOKENROUTER_API_KEY=sk-your-key-here
ENV_FILE = Path("data-extraction/.env")

def read_env(path):
    values = {}
    if not path.exists():
        return values
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values

API_KEY = os.environ.get("TOKENROUTER_API_KEY") or read_env(ENV_FILE).get("TOKENROUTER_API_KEY", "")
BASE_URL = "https://api.tokenrouter.com/v1"
MODEL = "moonshotai/kimi-k3-free"

if not API_KEY:
    raise SystemExit(
        f"TOKENROUTER_API_KEY not found.\n"
        f"Add this line to {ENV_FILE.resolve()}\n"
        f"    TOKENROUTER_API_KEY=sk-your-key-here"
    )

source = "shell" if os.environ.get("TOKENROUTER_API_KEY") else str(ENV_FILE)
print(f"key  : {API_KEY[:6]}...{API_KEY[-4:]} ({len(API_KEY)} chars, from {source})")
print(f"base : {BASE_URL}")
print(f"model: {MODEL}")

## 3. The vendor's example, with usage captured

This is TokenRouter's own snippet. One change: their loop asks for
`include_usage` but then discards it, because the usage arrives in a final
chunk that carries no `choices`. Token accounting is the whole point of
checking a token allowance, so it is captured here.

In [3]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

messages = [
    {"role": "system", "content": "You are an intelligent assistant, please reply concisely."},
    {"role": "user", "content": "Hello, what kind of model are you?"},
]

stream = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    stream=True,
    stream_options={"include_usage": True},
    extra_body={},
)

content_parts = []
usage = None
served_by = None

for chunk in stream:
    if getattr(chunk, "usage", None):      # final chunk, no choices
        usage = chunk.usage
    if getattr(chunk, "model", None):
        served_by = chunk.model
    if chunk.choices:
        delta = chunk.choices[0].delta
        if delta and delta.content:
            content_parts.append(delta.content)

full_content = "".join(content_parts)

print(full_content)
print("\n---")
print("model returned by the server:", served_by)
print("usage:", usage)

I’m Kimi, an AI assistant developed by Moonshot AI (月之暗面).

---
model returned by the server: kimi-k3
usage: CompletionUsage(completion_tokens=149, prompt_tokens=119, total_tokens=268, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=117, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=0), cached_tokens=0)


## 4. The test that decides it — structured classification

Streaming a greeting proves the endpoint answers. It does not prove the model
can do our job. This runs the real MUFASA rubric and requires valid JSON back.

In [4]:
import json
import time

CLASSIFY = '''You screen scientific papers for MUFASA.
Decide whether the scientific contribution is materially connected to Africa.
African authorship or an incidental study location alone is insufficient.
Score 0-4: african_centrality, local_specificity, scientific_depth, knowledge_value, local_applicability.
Return one minified JSON object only:
{"decision":"include|review|exclude","african_centrality":0,"local_specificity":0,"scientific_depth":0,"knowledge_value":0,"local_applicability":0,"reason":"at most 30 words"}

PAPER
Title: Rice husk ash as a partial cement replacement in Nigerian concrete
OpenAlex field: Engineering
OpenAlex topic: Construction materials
Abstract: Rice husk ash sourced from mills in Ogun State, Nigeria replaced 10
percent of ordinary Portland cement. Compressive strength reached 31.2 MPa at
28 days against 29.8 MPa for the control mix.'''

# max_tokens has to cover the reasoning AND the answer. On this same prompt
# the reasoning alone has measured 223 and 340 tokens on different runs, so
# 400 truncates the JSON on the longer ones and fails intermittently.
MAX_OUTPUT = 1024

started = time.perf_counter()
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": CLASSIFY}],
    temperature=0,
    max_tokens=MAX_OUTPUT,
)
elapsed = time.perf_counter() - started

choice = response.choices[0]
raw = choice.message.content or ""
usage = response.usage
details = getattr(usage, "completion_tokens_details", None)
reasoning = getattr(details, "reasoning_tokens", None) or 0

print("latency      :", round(elapsed, 2), "s")
print("finish_reason:", choice.finish_reason)
print("usage        :", usage)
print(f"reasoning    : {reasoning} of {usage.completion_tokens} output tokens "
      f"({reasoning / max(usage.completion_tokens, 1):.0%})")
print("raw          :", raw[:500])

if choice.finish_reason == "length":
    raise RuntimeError(
        f"Truncated: hit max_tokens={MAX_OUTPUT} with {reasoning} tokens spent "
        "reasoning, so the JSON never closed. Raise MAX_OUTPUT, or disable "
        "reasoning in cell 4b."
    )

cleaned = raw.replace("```json", "").replace("```", "").strip()
start, end = cleaned.find("{"), cleaned.rfind("}")
if start < 0 or end < start:
    raise ValueError(f"No complete JSON object in the reply: {cleaned[:200]!r}")

parsed = json.loads(cleaned[start:end + 1])
print("\nPARSED:", json.dumps(parsed, indent=2))

TOKENS_PER_PAPER = usage.total_tokens
SECONDS_PER_PAPER = elapsed

latency      : 5.44 s
finish_reason: stop
usage        : CompletionUsage(completion_tokens=237, prompt_tokens=290, total_tokens=527, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=155, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=0), cached_tokens=0)
reasoning    : 155 of 237 output tokens (65%)
raw          : {"decision":"include","african_centrality":4,"local_specificity":4,"scientific_depth":3,"knowledge_value":3,"local_applicability":4,"reason":"Locally sourced Nigerian rice husk ash valorized as cement replacement; directly addresses affordable, sustainable construction materials for Nigeria with verified strength performance."}

PARSED: {
  "decision": "include",
  "african_centrality": 4,
  "local_specificity": 4,
  "scientific_depth": 3,
  "knowledge_value": 3,
  "local_applicability": 4,
  "reason": "Locally sourced Ni

## 4b. Can the reasoning be turned off?

The baseline run spent **223 of 308 output tokens on reasoning** — 72% of the
output, on a task that is a rubric lookup. Chain-of-thought buys nothing here
and it is what makes a 6,000-paper run take a day.

Different gateways expose the switch differently and TokenRouter does not
document which it passes through, so this tries each. Unsupported ones will
error, which is a result, not a failure.

In [5]:
import time

VARIANTS = {
    "baseline (reasoning on)": {},
    "thinking.disabled":       {"thinking": {"type": "disabled"}},
    "reasoning_effort=none":   {"reasoning_effort": "none"},
    "reasoning_effort=minimal": {"reasoning_effort": "minimal"},
    "enable_thinking=False":   {"chat_template_kwargs": {"enable_thinking": False}},
}

def extract_json(text):
    """Return the parsed object, or None if the reply has no complete JSON."""
    cleaned = (text or "").replace("```json", "").replace("```", "").strip()
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start < 0 or end < start:
        return None
    try:
        return json.loads(cleaned[start:end + 1])
    except Exception:
        return None

rows = []
for label, extra in VARIANTS.items():
    started = time.perf_counter()
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": CLASSIFY}],
            temperature=0,
            max_tokens=MAX_OUTPUT,          # same budget as cell 4, so this is a fair test
            extra_body=extra,
        )
        elapsed = time.perf_counter() - started
        choice, usage = resp.choices[0], resp.usage
        details = getattr(usage, "completion_tokens_details", None)
        rows.append({
            "variant": label,
            "seconds": round(elapsed, 1),
            "total": usage.total_tokens,
            "output": usage.completion_tokens,
            "reasoning": getattr(details, "reasoning_tokens", None) or 0,
            "finish": choice.finish_reason,
            "valid_json": extract_json(choice.message.content) is not None,
            "error": "",
        })
    except Exception as exc:
        rows.append({
            "variant": label, "seconds": round(time.perf_counter() - started, 1),
            "total": None, "output": None, "reasoning": None, "finish": "-",
            "valid_json": False, "error": f"{type(exc).__name__}: {exc}"[:110],
        })
    print(f"  tried {label}")

import pandas as pd
display(pd.DataFrame(rows))

# Pick the cheapest variant that still returns parseable JSON.
usable = [r for r in rows if r["valid_json"] and r["total"]]
if usable:
    best = min(usable, key=lambda r: r["total"])
    BEST_EXTRA = VARIANTS[best["variant"]]
    TOKENS_PER_PAPER = best["total"]
    SECONDS_PER_PAPER = best["seconds"]
    baseline = next((r for r in rows if r["variant"].startswith("baseline")), None)
    print(f"\nBest usable: {best['variant']}")
    print(f"  {best['total']} tokens, {best['seconds']}s, {best['reasoning']} reasoning tokens")
    if baseline and baseline["total"] and best["total"] < baseline["total"]:
        saved = 1 - best["total"] / baseline["total"]
        faster = baseline["seconds"] / max(best["seconds"], 0.1)
        print(f"  vs baseline: {saved:.0%} fewer tokens, {faster:.1f}x faster")
    else:
        print("  Nothing beat the baseline - reasoning cannot be disabled here.")
else:
    BEST_EXTRA = {}
    print("\nNothing returned valid JSON. Keeping the baseline settings.")

# Any row showing finish='length' truncated: it ran out of budget mid-answer.
truncated = [r["variant"] for r in rows if r.get("finish") == "length"]
if truncated:
    print(f"\nTruncated (hit max_tokens): {truncated}")
    print("Those are budget failures, not quality failures.")

print("\nBEST_EXTRA =", BEST_EXTRA)

  tried baseline (reasoning on)
  tried thinking.disabled
  tried reasoning_effort=none
  tried reasoning_effort=minimal
  tried enable_thinking=False


,variant,seconds,total,output,reasoning,finish,valid_json,error
0,baseline (reasoning on),11.4,674.0,384.0,297.0,stop,True,
1,thinking.disabled,0.6,NaN,NaN,NaN,-,False,BadRequestError: Error code: 400 - {'error': {...
2,reasoning_effort=none,0.5,NaN,NaN,NaN,-,False,BadRequestError: Error code: 400 - {'error': {...
3,reasoning_effort=minimal,0.6,NaN,NaN,NaN,-,False,BadRequestError: Error code: 400 - {'error': {...
4,enable_thinking=False,2.0,300.0,77.0,0.0,stop,True,



Best usable: enable_thinking=False
  300 tokens, 2.0s, 0 reasoning tokens
  vs baseline: 55% fewer tokens, 5.7x faster

BEST_EXTRA = {'chat_template_kwargs': {'enable_thinking': False}}


## 4c. Throughput, and when to run

Token budget is not the constraint — throughput is, and throughput here is a
function of **what time you run it**.

`kimi-k3-free` is a shared pool on TokenRouter, and TokenRouter markets to
Cursor / Cline / Claude Code users, so the load is mostly Western developers.
The quiet window is Western night. Measured on the same prompt in one session:

| Condition | Per call |
|---|---|
| Congested | 125s, and once ~314s |
| Quiet, reasoning on | 5–22s |
| **Quiet (~04:00 UTC+1), `enable_thinking=False`** | **2–3s** |

The slow calls are not a penalty for bursting — calls immediately after a 125s
stall came back in 3s. They are queue position behind other users.

Best window observed: roughly **02:00–08:00 UTC** (03:00–09:00 in Nigeria),
when Europe is asleep and the US has wound down. Note this is *not* explained
by Chinese demand — 04:00 in Nigeria is midday in China, and the service was
at its fastest.

Levers in order: run in the quiet window, disable reasoning, cap the tail with
`TIMEOUT` plus a retry, and only then add workers. The timeout turns a 125s
stall into a 30s write-off rather than a stalled batch.

Any number this cell prints is only valid for the hour you measured it.
Re-check before committing to a bulk run.

In [ ]:
import concurrent.futures

# --- the dials ------------------------------------------------------------
WORKERS = 2       # change to 3, 4, 8... and re-run this cell
CALLS   = 4       # total requests to send
TIMEOUT = 30      # give up on a stalled call instead of waiting minutes
RETRIES = 1       # retry once; a stall is usually transient
# --------------------------------------------------------------------------

def one_call(_):
    t0 = time.perf_counter()
    last = ""
    for attempt in range(RETRIES + 1):
        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": CLASSIFY}],
                temperature=0,
                max_tokens=MAX_OUTPUT,
                extra_body=BEST_EXTRA,
                timeout=TIMEOUT,
            )
            good = extract_json(r.choices[0].message.content) is not None
            return time.perf_counter() - t0, good, ("retried" if attempt else "")
        except Exception as exc:
            last = type(exc).__name__
    return time.perf_counter() - t0, False, last

t0 = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as pool:
    out = list(pool.map(one_call, range(CALLS)))
wall = time.perf_counter() - t0

ok = sum(1 for _, good, _ in out if good)
retried = sum(1 for _, _, note in out if note == "retried")

print(f"{WORKERS} workers, {CALLS} calls -> {ok}/{CALLS} ok in {wall:.0f}s")
print(f"stalled and retried: {retried}/{CALLS}\n")
for i, (secs, good, note) in enumerate(out):
    print(f"  call {i}: {secs:6.1f}s   {'ok' if good else 'FAILED'}   {note}")

if ok:
    per_paper = wall / ok
    print(f"\n{per_paper:.1f}s per paper -> ~{6000 * per_paper / 3600:.1f}h for 6,000 papers")
    print("Latency here tracks load on the shared free pool, so this number is only")
    print("valid for the time of day you measured it. Re-check before a bulk run.")

## 5. What 50 million tokens would actually buy

Uses the measured token count from cell 4 rather than an assumption.

In [7]:
BUDGET = 50_000_000
CORPUS = 6_000

papers_affordable = BUDGET // TOKENS_PER_PAPER
needed = CORPUS * TOKENS_PER_PAPER

print(f"measured        : {TOKENS_PER_PAPER} tokens, {SECONDS_PER_PAPER:.1f}s per paper")
print(f"50M would buy   : ~{papers_affordable:,} papers")
print(f"MUFASA needs    : {CORPUS:,} papers = {needed/1e6:.1f}M tokens "
      f"({needed/BUDGET:.0%} of the allowance)")
print(f"serial runtime  : ~{CORPUS * SECONDS_PER_PAPER / 3600:.1f}h "
      f"(divide by concurrency)")
print()
print("Enough." if needed <= BUDGET else "NOT enough — the corpus exceeds the allowance.")

measured        : 300 tokens, 2.0s per paper
50M would buy   : ~166,666 papers
MUFASA needs    : 6,000 papers = 1.8M tokens (4% of the allowance)
serial runtime  : ~3.3h (divide by concurrency)

Enough.


## 6. Rate limits and remaining quota

The advertised figure means nothing if per-minute limits make it unusable, or
if the balance never actually moves. Headers are where the truth is.

In [8]:
import requests

r = requests.get(
    f"{BASE_URL}/models",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30,
)
print("GET /models ->", r.status_code)

limits = {k: v for k, v in r.headers.items()
          if any(w in k.lower() for w in ("ratelimit", "quota", "credit", "remaining", "reset"))}
print("quota/rate headers:", limits or "none advertised")

if r.status_code == 200:
    ids = [m.get("id") for m in r.json().get("data", [])]
    print(f"models offered: {len(ids)}")
    print("kimi variants :", [i for i in ids if "kimi" in str(i).lower()] or "none")
    print("free-tagged   :", [i for i in ids if "free" in str(i).lower()][:10] or "none")

c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


GET /models -> 200
quota/rate headers: none advertised
models offered: 120
kimi variants : ['moonshotai/kimi-k3-free', 'moonshotai/kimi-k3', 'moonshotai/kimi-k2.7-code', 'moonshotai/kimi-k2.6', 'moonshotai/kimi-k2.5']
free-tagged   : ['moonshotai/kimi-k3-free', 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free']


## What to conclude

- **Cell 3 fails** → the offer is not usable at all.
- **Cell 3 works, cell 4 fails** → answers greetings but not our rubric; would
  need prompt work before it is worth anything.
- **Both work** → viable for corpus classification. Check the quota page
  before and after this run. If the balance does not move, the 50M number is
  decorative.

**Do not make the pipeline depend on this.** The post itself says to use it
"before they change or remove the free plan." Keep the local llama.cpp path
working so the corpus can still be built if this disappears mid-run.

If it does not hold up: OpenRouter publishes genuinely free models with stated
limits, and Gemini's free tier is the fallback already named in the retrieval
milestone. Both are OpenAI-compatible — change `BASE_URL` and `MODEL` and
every cell here still works.